In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from pathlib import Path
from copy import deepcopy
import multiprocessing as mp
import logging
import random
from itertools import cycle
import os
from functools import partial

import mlflow
import matplotlib.pyplot as plt
from networkx import number_weakly_connected_components
import torch
import numpy as np
from dotenv import load_dotenv

from halcyon.ModelRunner import run_training_job, ModelRunner
from halcyon.Simulators import NoiseLevel, SimulationResult

from halcyon.Models import PhoenixAddAndProdLayers, MLP, CellBox


seed = 174
# for CPU
torch.manual_seed(seed)              
# for all GPUs
torch.cuda.manual_seed_all(seed)     
np.random.seed(seed) 
random.seed(seed) 

load_dotenv() 
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s',
    force=True  # Important in Jupyter to reset config
)


Check if MLFlow connects correctly

In [2]:
print(os.getenv("MLFLOW_TRACKING_URI"))

https://mlflow.cropresilience.org


In [3]:
[f"{experiment.name=}" for experiment in mlflow.search_experiments()]
mlflow.set_experiment("Halcyon sandbox experiment")
mlflow.set_experiment_tag("Tool name", "HALCYON") 

# Input data

## Downloading data from MLFlow

In [4]:
sim_run_id_to_use = "8f557eb8aadd45fdba24175cc74672c3"  
run = mlflow.get_run(sim_run_id_to_use)

params = run.data.params
print(params)

{'nr_genes': '20', 'sparsity': '0.1', 'num_modules': '2', 'noise_level': 'LOW', 'seed': '174'}


In [5]:
ml_flow_download_path = mlflow.artifacts.download_artifacts(run_id=sim_run_id_to_use, artifact_path="dataset")
sim_output = SimulationResult.load_from_pkl(Path(ml_flow_download_path) / "simulation_result.pkl")

2026-03-17 15:56:33,662 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-03-17 15:56:33,673 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-03-17 15:56:33,685 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-03-17 15:56:33,687 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-03-17 15:56:33,695 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-03-17 15:56:33,704 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-03-17 15:56:33,711 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-03-17 15:56:33,724 - WARNING - Connection p

## Convert sim output into datasets for use with the trainer object

In [6]:
# From sim output to datasets
single_ko_dataset, double_ko_dataset, time_scaling_factor = sim_output.create_datasets()
train_ko_dataset, validation_ko_dataset = single_ko_dataset.split_train_test(test_fraction=0.25)

train_dataset_name = "train_single_ko"
val_dataset_name ="val_single_ko"
test_dataset_name = "double_ko"
dataset_dict = {
    train_dataset_name: train_ko_dataset,
    val_dataset_name: validation_ko_dataset,
    test_dataset_name: double_ko_dataset,}

for name, dataset in dataset_dict.items():
    print(f"Length {name}: {len(dataset)}")

Length train_single_ko: 15
Length val_single_ko: 5
Length double_ko: 20


# Trainers (with different models) creation

In [14]:
train_jobs : list[ModelRunner] = []
device = cycle(['cuda:0', 'cuda:1'])
batch_size = 32
nr_epochs = 20

## Phoenix

In [8]:
for hidden_size in [10]:
    for do_only_pos in [True]:        
        ml_model = PhoenixAddAndProdLayers(
                    modelled_gene_labels=single_ko_dataset.genes,
                    hidden_size=hidden_size,
                    condition_names=single_ko_dataset.all_environments,
                    synthesis_only_positive=do_only_pos,
                    device=next(device)
            )

        # my_loss = MSELossPlusMachineLearningL2(lambda_param=0.5)
        model_runner = ModelRunner(
            model = ml_model,
            batch_size=batch_size,
            optimizer_kwargs = {'lr': 0.01}
        )
        train_jobs.append(model_runner)

## CellBox

In [15]:
cellbox = CellBox(
            modelled_gene_labels=single_ko_dataset.genes,
            condition_names=single_ko_dataset.all_environments,
            device=next(device)
    )

model_runner = ModelRunner(
    model = cellbox,
    batch_size=batch_size,
    optimizer_kwargs = {'lr': 0.01}
)
train_jobs.append(model_runner)

## MLP

In [23]:
mlp = MLP(
        modelled_gene_labels=single_ko_dataset.genes,
        condition_names=single_ko_dataset.all_environments,
        hidden_size=5,
        nr_layers=2,
        device=next(device)
)

model_runner = ModelRunner(
    model = mlp,
    batch_size=batch_size,
    optimizer_kwargs = {'lr': 0.01}
)
train_jobs.append(model_runner)

In [24]:
for job in train_jobs:
    print(job)
    print()

Modelrunner (Model=CellBox with hyperparams: ['device=cuda:0', 'phi=tanh', 'self_activations=False'],
 optimizer=Adam,
 optimizer_kwargs={'lr': 0.01},
 batch_size=32
)

Modelrunner (Model=MLP with hyperparams: ['device=cuda:1', 'hidden_size=5', 'synthesis_only_positive=False'],
 optimizer=Adam,
 optimizer_kwargs={'lr': 0.01},
 batch_size=32
)



In [25]:
train_jobs[1]

Modelrunner (Model=MLP with hyperparams: ['device=cuda:1', 'hidden_size=5', 'synthesis_only_positive=False'],
 optimizer=Adam,
 optimizer_kwargs={'lr': 0.01},
 batch_size=32
)

In [26]:
train_jobs[1].train_and_predict_for_mlflow(dataset_dict=dataset_dict, 
                                           train_data_key_in_dict=train_dataset_name,
                                           val_data_key_in_dict=val_dataset_name,
                                           test_data_key_in_dict=test_dataset_name,
                                           nr_epochs=nr_epochs,
                                           simulation_parent_id=sim_run_id_to_use,
                                           save_every_n_epochs=10)

2026-03-17 16:16:41,524 [MainProcess] Epoch 0, train_loss	0.13451898097991943
2026-03-17 16:16:43,716 [MainProcess] Epoch 0, val_loss	0.1167687177658081
2026-03-17 16:16:49,966 [MainProcess] Epoch 0, test_loss	0.08830313384532928
2026-03-17 16:16:56,575 [MainProcess] Epoch 1, train_loss	0.12285520881414413
2026-03-17 16:16:59,016 [MainProcess] Epoch 1, val_loss	0.10588270425796509
2026-03-17 16:17:03,874 [MainProcess] Epoch 1, test_loss	0.08570986986160278
2026-03-17 16:17:12,227 [MainProcess] Epoch 2, train_loss	0.11207757145166397
2026-03-17 16:17:14,368 [MainProcess] Epoch 2, val_loss	0.09572301805019379
2026-03-17 16:17:20,891 [MainProcess] Epoch 2, test_loss	0.08402646332979202
2026-03-17 16:17:26,691 [MainProcess] Epoch 3, train_loss	0.10209332406520844
2026-03-17 16:17:28,273 [MainProcess] Epoch 3, val_loss	0.08634945750236511
2026-03-17 16:17:34,939 [MainProcess] Epoch 3, test_loss	0.08322664350271225
2026-03-17 16:17:40,566 [MainProcess] Epoch 4, train_loss	0.09289699792861938

🏃 View run MLP at: https://mlflow.cropresilience.org/#/experiments/3/runs/badb9fbcd5154fc89eb313035c60cfd3
🧪 View experiment at: https://mlflow.cropresilience.org/#/experiments/3


# Do training

In [ ]:
# mp.set_start_method("spawn", force=True)
# with mp.Pool(processes=4) as pool:
#     pool.map(
#         partial(
#             run_training_job, 
#             dataset_dict=dataset_dict, 
#             train_data_key_in_dict=train_dataset_name, 
#             simulation_parent_id=sim_run_id_to_use, 
#             do_mlflow_model_logging=False,
#         ),
#         train_jobs
#     )

# All files have now been logged to MLFlow

To see how to process this data further look into the MLFlow loading and plotting notebook(s)